In [ ]:
#Libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, PchipInterpolator

#------------------------------------------------
#Constants 
Msun_in_km = 1.474  #In units where G=c=1 (Geometrized)
hbar_c = 197.3269804  #MeV·fm
MeVfm3_to_Jm3 = 1.6022e32 
Jm3_to_1km2 = 8.234568e-39 
MeVfm3_to_1km2 = MeVfm3_to_Jm3 * Jm3_to_1km2

#P,en: MeV/fm3 
#P_g,en_g: 1/km2

#Quark pressure
def P_q(a4, D, Beff_14, xm): 
    Beff = Beff_14**4  #(B_eff^(1/4))
    P_q_g = 3/(4*(np.pi**2)) * a4 * ((xm/3)**4) + 3/((np.pi**2)) * (D**2) * (xm/3)**2 - Beff
    P_q = P_q_g /(hbar_c**3)         #MeV/fm3
    return P_q

#Quark energy density
def den_en_xm_q(a4, D, Beff_14, xm):
    Beff = Beff_14**4
    den_en_q_g =  9/(4*(np.pi**2)) * a4 * ((xm/3)**4) + 3/((np.pi**2)) * (D**2) * (xm/3)**2 + Beff
    den_en_q = den_en_q_g /(hbar_c**3)         #MeV/fm3
    return den_en_q

#Construct ε(P) from Ρ(μ) and ε(μ)
def build_den_en_q_of_P(a4, D, Beff_14, xm_min=800, xm_max=2500, N=8000):
    
    xm_grid = np.linspace(xm_min, xm_max, N)
    P_grid  = P_q(a4, D, Beff_14, xm_grid)
    den_en_grid  = den_en_xm_q(a4, D, Beff_14, xm_grid)

    #Sort with respect to pressure (ascending)
    idx = np.argsort(P_grid)  #Indices that sort P_grid in ascending order
    P_sorted = P_grid[idx]
    xm_sorted = xm_grid[idx]
    den_en_sorted = den_en_grid[idx]

    #Remove duplicate pressure values
    P_unique, idx_unique = np.unique(P_sorted, return_index=True)
    xm_unique = xm_sorted[idx_unique]
    den_en_unique  = den_en_sorted[idx_unique]

    #μ(Ρ) and ε(μ)
    xm_of_P = PchipInterpolator(P_unique, xm_unique, extrapolate=False)      #Returns NaN for values outside the data range
    den_en_of_xm = PchipInterpolator(xm_unique, den_en_unique, extrapolate=False)

    def den_en_q_of_P(P):   #ε(Ρ)
        xm = xm_of_P(P)   #μ(Ρ)
        return den_en_of_xm(xm) #ε(μ(Ρ))=ε(Ρ)

    return den_en_q_of_P, (P_unique.min(), P_unique.max())  